# Projection maps


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

start_year = 2025
end_year = 2100
EVAL6_DIR = Path("../../results/model/projection/eval6")
OUT_DIR = Path("../../results/si_figures/si_fig6-7")
MODELS = ["CDD", "CDDP", "SIAM", "SIAMP"]

def read_eos_projection(ssp):
        fp = EVAL6_DIR / f"eval6_{ssp}_predictions.pkl"
    if not fp.exists():
        raise FileNotFoundError(
            f"Missing {fp}. Run run_eval6_{ssp}_eos_through_2100.py first."
        )
    eos_projection = pd.read_pickle(fp)
    years = eos_projection[0].get("years") if eos_projection else None
    print(f"Loaded {fp}")
    print(f"n records={len(eos_projection)} | models={sorted({r['model'] for r in eos_projection})}")
    n_pix = len({(r['latitude'], r['longitude']) for r in eos_projection})
    print(f"n grids={n_pix}")
    if years is not None:
        print(f"years={int(min(years))}–{int(max(years))} (n={len(years)})")
    return eos_projection

def eos_change_summary_with_best(eos_projection, window=10):
    rows = []
    for rec in eos_projection:
        arr = rec.get("predicted_eos")
        if arr is None or len(arr) < window * 2:
            diff = np.nan
        else:
            first_vals = arr[:window].astype(float)
            last_vals = arr[-window:].astype(float)
            diff = np.nanmean(last_vals) - np.nanmean(first_vals)
        rows.append({
            "latitude": rec.get("latitude"),
            "longitude": rec.get("longitude"),
            "model": rec.get("model"),
            "predicted_eos_change": diff,
        })
    diff_df = pd.DataFrame(rows)
    lowest_diff_df = (
        diff_df.loc[
            diff_df.groupby(["latitude", "longitude"])["predicted_eos_change"].idxmin()
        ]
        .reset_index(drop=True)
    )
    return diff_df, lowest_diff_df


In [ ]:
## Lowest ΔEOS map
import matplotlib.pyplot as plt
import rasterio as rs
from rasterio.features import rasterize
from rasterio.transform import from_origin
import geopandas as gpd
import numpy as np
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from shapely.geometry import box
import matplotlib.path as mpath
from matplotlib.colors import ListedColormap

def rasterize_best_model(gdf, transform, width, height, model_classes, metric):
    shapes = (
        (geom, model_classes[model])
        for geom, model in zip(gdf.geometry, gdf[metric])
    )
    model_raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype="float32",
    )
    return model_raster

def show_best_model_map(
    df, metric,
    tiff_path="../../data/satellite_data/images/base-image/test.tif",
    mode="scale_xy",            # "absolute", "scale", or "scale_xy"
    res_x=0.25,                 # used for "absolute": pixel width in CRS units
    res_y=0.25,                 # used for "absolute": pixel height in CRS units
    scale_factor=2.0,           # used for "scale": same factor for both axes
    scale_factor_x=12.5,        # used for "scale_xy": longitude (x) factor
    scale_factor_y=9.4,         # used for "scale_xy": latitude (y) factor
    continents=None             # e.g., ["North America", "Europe"]; None = all land
):
    with rs.open(tiff_path) as src:
        crs = src.crs
        left, bottom, right, top = src.bounds
        transform0 = src.transform

    orig_res_x = transform0.a
    orig_res_y = abs(transform0.e)

    if mode == "absolute":
        px_x = res_x
        px_y = res_y
    elif mode == "scale":
        px_x = orig_res_x * scale_factor
        px_y = orig_res_y * scale_factor
    elif mode == "scale_xy":
        px_x = orig_res_x * scale_factor_x
        px_y = orig_res_y * scale_factor_y
    else:
        raise ValueError("mode must be 'absolute', 'scale', or 'scale_xy'")

    width = int(np.ceil((right - left) / px_x))
    height = int(np.ceil((top - bottom) / px_y))
    transform = from_origin(left, top, px_x, px_y)

    print(f"[DEBUG] Original pixel size: x={orig_res_x}, y={orig_res_y} (CRS units)")
    print(f"[DEBUG] New pixel size: x={px_x}, y={px_y} (CRS units)")
    print(f"[DEBUG] Output grid: width={width}, height={height}")

    model_colors = {
        'CDD': '#A3D4E0',
        'CDDP': '#5BAED8',
        'SIAM': '#3B83C4',
        'SIAMP': '#D1837D',
    }
    model_classes = {name: i for i, name in enumerate(model_colors.keys())}

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs=crs
    )

    raster_bounds = box(left, bottom, right, top)
    gdf = gdf[gdf.geometry.within(raster_bounds)]
    print(f"[DEBUG] Points inside raster bounds: {len(gdf)}")

    model_raster = rasterize_best_model(gdf, transform, width, height, model_classes, metric)

    ne_path = shpreader.natural_earth(
        resolution="110m",
        category="cultural",
        name="admin_0_countries"
    )
    countries = gpd.read_file(ne_path)  # CRS: EPSG:4326

    if continents is not None:
        countries = countries[countries["CONTINENT"].isin(continents)]

    land_poly = countries.union_all()
    land_gdf = gpd.GeoDataFrame(geometry=[land_poly], crs="EPSG:4326")

    land_gdf = land_gdf.to_crs(crs)

    land_gdf = gpd.clip(land_gdf, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))

    land_mask = rasterize(
        [(geom, 1) for geom in land_gdf.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
    )

    model_raster = np.where(land_mask == 1, model_raster, np.nan)

    fig = plt.figure(figsize=[6, 6])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], ccrs.PlateCarree())
    ax.coastlines()

    theta = np.linspace(0, 2 * np.pi, 100)
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    circle = mpath.Path(verts * 0.5 + [0.5, 0.5])
    ax.set_boundary(circle, transform=ax.transAxes)

    cmap = ListedColormap([model_colors[m] for m in model_classes.keys()])
    cmap.set_bad((0, 0, 0, 0))  # Make NaN transparent

    ax.imshow(
        np.ma.masked_invalid(model_raster),
        cmap=cmap,
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        origin="upper",
        interpolation="nearest",
    )

    import matplotlib.ticker as mticker
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    gl = ax.gridlines(linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])
    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30, 50, 70]:
        ax.text(180, lat + 5, lat_formatter(lat),
                transform=ccrs.PlateCarree(), ha='left', va='center',
                fontsize=12, color='black')
    for lon in np.arange(-180, 181, 60):
        if lon == -180 or lon == -60:  # skip -180 (already done) and -60 (60°W)
            continue
    
        label_lon, label_lat = lon, 25  # adjust position
    
        if lon in [-120, 120]:
            label_lat = 23
        if lon == 0:
            label_lon = 2
            label_lat = 29
        if lon == 180:
            label_lon = 178
    
        ax.text(label_lon, label_lat, lon_formatter(lon),
                transform=ccrs.PlateCarree(),
                ha='center', va='top', fontsize=12, color='black')
        
    valid_pixels = np.isfinite(model_raster).sum()
    percentages = []
    
    for model, idx in model_classes.items():
        count = np.sum(model_raster == idx)
        pct = (count / valid_pixels) * 100 if valid_pixels > 0 else 0
        percentages.append(pct)
    
    inset_ax = fig.add_axes([0.12, 0.10, 0.35, 0.25])  # [left, bottom, width, height]
    
    inset_ax.bar(
        model_classes.keys(),
        percentages,
        color=[model_colors[m] for m in model_classes.keys()],
        edgecolor="none"
    )
    
    inset_ax.tick_params(axis="x", labelrotation=45, labelsize=10)
    inset_ax.tick_params(axis="y", labelsize=10)
    inset_ax.set_ylabel("Percentage (%)", fontsize=10)

    inset_ax.spines['top'].set_visible(False)
    inset_ax.spines['right'].set_visible(False)

    # plt.tight_layout()
    # plt.show()
    return fig


In [ ]:
## ΔEOS by model
import matplotlib.pyplot as plt
import rasterio as rs
from rasterio.features import rasterize
from rasterio.transform import from_origin
import geopandas as gpd
import numpy as np
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader
from shapely.geometry import box
import matplotlib.path as mpath
from matplotlib.colors import Normalize, LinearSegmentedColormap

def show_eos_change_map(
    df, model,
    tiff_path="../../data/satellite_data/images/base-image/test.tif",
    mode="scale_xy",
    res_x=0.25,
    res_y=0.25,
    scale_factor=2.0,
    scale_factor_x=12.5,
    scale_factor_y=9.4,
    continents=None,
    vmin=-7,
    vmax=17,
):
    with rs.open(tiff_path) as src:
        crs = src.crs
        left, bottom, right, top = src.bounds
        transform0 = src.transform

    orig_res_x = transform0.a
    orig_res_y = abs(transform0.e)

    if mode == "absolute":
        px_x, px_y = res_x, res_y
    elif mode == "scale":
        px_x, px_y = orig_res_x * scale_factor, orig_res_y * scale_factor
    elif mode == "scale_xy":
        px_x, px_y = orig_res_x * scale_factor_x, orig_res_y * scale_factor_y
    else:
        raise ValueError("mode must be 'absolute', 'scale', or 'scale_xy'")

    width = int(np.ceil((right - left) / px_x))
    height = int(np.ceil((top - bottom) / px_y))
    transform = from_origin(left, top, px_x, px_y)

    df_model = df[df["model"] == model].copy()
    gdf = gpd.GeoDataFrame(
        df_model,
        geometry=gpd.points_from_xy(df_model.longitude, df_model.latitude),
        crs=crs
    )

    raster_bounds = box(left, bottom, right, top)
    gdf = gdf[gdf.geometry.within(raster_bounds)]
    print(f"[DEBUG] Pixels plotted: {len(gdf)}")

    shapes = list(zip(gdf.geometry, gdf["predicted_eos_change"]))
    val_raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=np.nan,
        dtype="float32"
    )

    ne_path = shpreader.natural_earth(
        resolution="110m", category="cultural", name="admin_0_countries"
    )
    countries = gpd.read_file(ne_path)
    if continents is not None:
        countries = countries[countries["CONTINENT"].isin(continents)]

    land_poly = countries.union_all()
    land = gpd.GeoDataFrame(geometry=[land_poly], crs="EPSG:4326").to_crs(crs)
    land = gpd.clip(land, gpd.GeoDataFrame(geometry=[raster_bounds], crs=crs))
    land_mask = rasterize(
        [(geom, 1) for geom in land.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8"
    )
    val_raster = np.where(land_mask == 1, val_raster, np.nan)

    colors = [
        "#0b3c68", "#165188", "#2066a8", "#4d91c4", "#8ec1da",
        "#fbebe1", "#f6d6c2", "#d47264", "#c14d48", "#ae282c"
    ]
    cmap = LinearSegmentedColormap.from_list("eos_change", colors)
    cmap.set_bad((0, 0, 0, 0))
    norm = Normalize(vmin=vmin, vmax=vmax)

    val_raster = np.clip(val_raster, vmin, vmax)

    fig = plt.figure(figsize=[7, 7])
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.NorthPolarStereo())
    ax.set_extent([-180, 180, 30, 90], crs=ccrs.PlateCarree())
    ax.coastlines()

    theta = np.linspace(0, 2 * np.pi, 100)
    circle = mpath.Path(
        np.vstack([np.sin(theta), np.cos(theta)]).T * 0.5 + 0.5
    )
    ax.set_boundary(circle, transform=ax.transAxes)

    im = ax.imshow(
        np.ma.masked_invalid(val_raster),
        cmap=cmap,
        norm=norm,
        extent=[left, right, bottom, top],
        transform=ccrs.PlateCarree(),
        origin="upper",
        interpolation="nearest"
    )

    import matplotlib.ticker as mticker
    from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
    gl = ax.gridlines(linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 60))
    gl.ylocator = mticker.FixedLocator([30, 50, 70])

    lat_formatter = LatitudeFormatter()
    lon_formatter = LongitudeFormatter()
    for lat in [30, 50, 70]:
        ax.text(180, lat + 5, lat_formatter(lat),
                transform=ccrs.PlateCarree(), fontsize=12)
    for lon in np.arange(-180, 181, 60):
        if lon == -180:
            continue
        ax.text(lon, 25, lon_formatter(lon),
                transform=ccrs.PlateCarree(), fontsize=12, ha='center')

    cbar = fig.colorbar(
        im,
        ax=ax,
        orientation="horizontal",
        pad=0.10,
        fraction=0.05,
    )
    cbar.set_label("EOS Change (days)", fontsize=14)
        tick0 = int(np.ceil(vmin / 5.0) * 5)
    tick1 = int(np.floor(vmax / 5.0) * 5)
    cbar.set_ticks(np.arange(tick0, tick1 + 1, 5))
    cbar.ax.tick_params(labelsize=12)

    plt.tight_layout()
    return fig



## SSP245


In [ ]:
ssp = "ssp245"
eos_projection = read_eos_projection(ssp)
diff_df, lowest_diff_df = eos_change_summary_with_best(eos_projection, window=10)
print(diff_df.groupby("model")["predicted_eos_change"].agg(["mean", "std", "min", "max"]).round(2))
print("lowest-ΔEOS model counts:\n", lowest_diff_df["model"].value_counts())

out_ssp = OUT_DIR / ssp
out_ssp.mkdir(parents=True, exist_ok=True)

fig = show_best_model_map(lowest_diff_df, "model")
fig.savefig(out_ssp / "lowest_diff.png", dpi=300, bbox_inches="tight")
print("Saved", out_ssp / "lowest_diff.png")

for model in MODELS:
    fig = show_eos_change_map(diff_df, model=model)
    fp = out_ssp / f"{model}.png"
    fig.savefig(fp, dpi=300, bbox_inches="tight")
    print("Saved", fp)


## SSP585


In [ ]:
ssp = "ssp585"
eos_projection = read_eos_projection(ssp)
diff_df, lowest_diff_df = eos_change_summary_with_best(eos_projection, window=10)
print(diff_df.groupby("model")["predicted_eos_change"].agg(["mean", "std", "min", "max"]).round(2))
print("lowest-ΔEOS model counts:\n", lowest_diff_df["model"].value_counts())

out_ssp = OUT_DIR / ssp
out_ssp.mkdir(parents=True, exist_ok=True)

fig = show_best_model_map(lowest_diff_df, "model")
fig.savefig(out_ssp / "lowest_diff.png", dpi=300, bbox_inches="tight")
print("Saved", out_ssp / "lowest_diff.png")

for model in MODELS:
    fig = show_eos_change_map(diff_df, model=model, vmax=32)
    fp = out_ssp / f"{model}.png"
    fig.savefig(fp, dpi=300, bbox_inches="tight")
    print("Saved", fp)

